In [21]:
# from google.colab import drive
# drive.mount('/content/drive')

In [22]:
import os
import pickle
import pandas as pd
import numpy as np
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random
from itertools import product
from typing import List, Tuple, Optional
from IPython.display import clear_output
%matplotlib inline

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

# desktop version
data_folder = "G:\Meu Drive\mestrado_final_files\data\\"
models_folder = "G:\Meu Drive\mestrado_final_files\models\\NGramModel_ExtraInfo\\"

# google colab version
# data_folder = "drive/MyDrive/mestrado_final_files/data/"
# models_folder = "drive/MyDrive/mestrado_final_files/models/NGramModel_ExtraInfo/"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [23]:
device

device(type='cpu')

In [24]:
pd.__version__

'2.2.3'

In [25]:
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)

In [26]:
BATCH_SIZE = 1024
EPOCHS = 50
VOCAB_SIZE = 601
TIME_SIZE = 301

run_models = 1

In [27]:
%run 01_functions_training.ipynb
#%run drive/MyDrive/mestrado_final_files/notebooks/01_functions_training.ipynb

In [28]:
with open(data_folder + 'data_token_features.pkl', 'rb') as f:
    df_token_features = pickle.load(f)

In [29]:
with open(data_folder + 'dict_set_covariables.pkl', 'rb') as f:
    dict_set_covariables = pickle.load(f)

In [30]:
list_seasons = ['2018-19', '2019-20', '2020-21', '2021-22', '2022-23']
list_context = [5]
list_embedding_layer_dim = [32, 64]
list_num_neurons = [50]
list_num_layers = [3, 5]
list_dropout = [0, 0.3]
list_layernorm = [False, True]
list_set_covariables = [1, 2]

In [31]:
model_mask_name_total = "{}_seasondata{}_context{}_embedding{}_numlayers{}_numneurons{}_{}_layernorm{}_setcovariables{}_parameters.pth"
model_mask_name_46min = "{}_first46m_seasondata{}_context{}_embedding{}_numlayers{}_numneurons{}_{}_layernorm{}_setcovariables{}_parameters.pth"
model_mask_name_2min = "{}_last2m_seasondata{}_context{}_embedding{}_numlayers{}_numneurons{}_{}_layernorm{}_setcovariables{}_parameters.pth"

In [32]:
all_combinations = list(product(list_seasons, list_context, list_embedding_layer_dim, list_num_neurons, list_num_layers,
                                list_dropout, list_layernorm, list_set_covariables))

In [33]:
len(all_combinations)

160

In [34]:
i = 0
for comb in all_combinations:
    i += 1
    print(i)
    season, CONTEXT_SIZE, EMBEDDING_LAYER_SIZE, NUM_HIDDEN_NEURONS, NUM_LAYERS, dropout_rate, layer_norm, key_covariables = comb

    print(comb)
    index_covariables = dict_set_covariables["index"][key_covariables]
    covariables = dict_set_covariables["covariables"][key_covariables]

    # ================== DADOS TREINO/DEV - MODELO TOTAL ==================
    df_ = df_token_features.query("season == @season & season_split == 'train'")

    trn_tkn = torch.tensor(np.asarray(df_["token_features"].to_list()), dtype = torch.int64, device = device)
    trn_num = torch.tensor(np.asarray(df_["numerical_features"].to_list()), dtype = torch.float32, device = device)
    trn_y   = torch.tensor(np.asarray(df_["target_tokens"].to_list()), dtype = torch.int64, device = device)
    trn_mask = torch.tensor(build_logit_mask_ngram_models(df_), dtype = torch.bool, device = device)

    df_ = df_token_features.query("season == @season & season_split == 'dev'")

    dev_tkn = torch.tensor(np.asarray(df_["token_features"].to_list()), dtype = torch.int64, device = device)
    dev_num = torch.tensor(np.asarray(df_["numerical_features"].to_list()), dtype = torch.float32, device = device)
    dev_y   = torch.tensor(np.asarray(df_["target_tokens"].to_list()), dtype = torch.int64, device = device)
    dev_mask = torch.tensor(build_logit_mask_ngram_models(df_), dtype = torch.bool, device = device)

    del df_
    # =======================================================================

    tp_dropout = "nodropout" if dropout_rate == 0 else "dropout" + str(dropout_rate).replace(".", "")

    token_model = NGramModel_ExtraInfo(
        context_size = CONTEXT_SIZE,
        vocab_size = VOCAB_SIZE,
        embedding_dim = EMBEDDING_LAYER_SIZE,
        numerical_input_dim = len(index_covariables),
        num_hidden_neurons = NUM_HIDDEN_NEURONS,
        num_layers = NUM_LAYERS,
        activation = nn.GELU,
        dropout_rate = dropout_rate,
        use_layer_norm = layer_norm)

    str_season = season.replace("-", "")

    model_filename = model_mask_name_total\
    .format(token_model.__class__.__name__, str_season, CONTEXT_SIZE, EMBEDDING_LAYER_SIZE, NUM_LAYERS, NUM_HIDDEN_NEURONS, tp_dropout, str(layer_norm), key_covariables)

    results_filename = "results_"  + model_filename.replace("_parameters.pth", ".pkl")

    if results_filename in os.listdir(models_folder):
        print("Already fitted.")
        continue

    # ================== TREINO MODELO TOTAL ==================
    token_model, trnloss, devloss = train_num_ngram_model(
        model = token_model,
        train_token_data = trn_tkn[:, -CONTEXT_SIZE:],
        train_numerical_data = trn_num[:, index_covariables],
        train_targets = trn_y,
        train_mask_data = trn_mask,
        batch_size = BATCH_SIZE,
        n_epochs = EPOCHS,
        lr = 0.002,
        val_token_data = dev_tkn[:, -CONTEXT_SIZE:],
        val_numerical_data = dev_num[:, index_covariables],
        val_targets = dev_y,
        val_mask_data = dev_mask,
        params_optimizer = {'factor': 0.5, 'threshold': 0, 'patience': 1},
        device = device)

    df_importance = permutation_importance_ngram(
        model = token_model,
        target = dev_y,
        tokens = dev_tkn[:, -CONTEXT_SIZE:],
        numerical_features = dev_num[:, index_covariables],
        names_numerical_features = covariables,
        num_repeats = 10,
        mask_data = dev_mask)

    print(df_importance)

    output = token_model(trn_tkn[:, -CONTEXT_SIZE:], trn_num[:, index_covariables])
    output = output.masked_fill(trn_mask, -1e9)
    baseline_loss = F.cross_entropy(output, trn_y).item()

    print(f"train data loss: {baseline_loss}")

    torch.save(token_model.state_dict(), models_folder + model_filename)

    dict_results = {
        "class_model": token_model.__class__.__name__,
        "model_filename": model_filename,
        "token_context_size": CONTEXT_SIZE,
        "token_embedding_dim": EMBEDDING_LAYER_SIZE,
        "training_epochs": EPOCHS,
        "numerical_covariables": covariables,
        "training_batch_loss": np.asarray(trnloss),
        "train_loss": baseline_loss,
        "dev_loss": np.asarray(devloss),
        "num_parameters": get_num_parameters(token_model),
        "variable_importance": df_importance}

    with open(models_folder + results_filename, 'wb') as f:
        pickle.dump(dict_results, f)

    #######################################################################
    # ================== MODELO 46 MINUTOS (FINE-TUNING) ==================
    df_ = df_token_features.query("season == @season & season_split == 'train' & ind_last_2_minutes_game == 0")

    trn_tkn = torch.tensor(np.asarray(df_["token_features"].to_list()), dtype = torch.int64, device = device)
    trn_num = torch.tensor(np.asarray(df_["numerical_features"].to_list()), dtype = torch.float32, device = device)
    trn_y   = torch.tensor(np.asarray(df_["target_tokens"].to_list()), dtype = torch.int64, device = device)
    trn_mask = torch.tensor(build_logit_mask_ngram_models(df_), dtype = torch.bool, device = device)

    df_ = df_token_features.query("season == @season & season_split == 'dev' & ind_last_2_minutes_game == 0")

    dev_tkn = torch.tensor(np.asarray(df_["token_features"].to_list()), dtype = torch.int64, device = device)
    dev_num = torch.tensor(np.asarray(df_["numerical_features"].to_list()), dtype = torch.float32, device = device)
    dev_y   = torch.tensor(np.asarray(df_["target_tokens"].to_list()), dtype = torch.int64, device = device)
    dev_mask = torch.tensor(build_logit_mask_ngram_models(df_), dtype = torch.bool, device = device)

    del df_

    token_model_46m = NGramModel_ExtraInfo(
        context_size = CONTEXT_SIZE,
        vocab_size = VOCAB_SIZE,
        embedding_dim = EMBEDDING_LAYER_SIZE,
        numerical_input_dim = len(index_covariables),
        num_hidden_neurons = NUM_HIDDEN_NEURONS,
        num_layers = NUM_LAYERS,
        activation = nn.GELU,
        dropout_rate = dropout_rate,
        use_layer_norm = layer_norm)

    str_season = season.replace("-", "")

    model_filename = model_mask_name_46min\
    .format(token_model_46m.__class__.__name__, str_season, CONTEXT_SIZE, EMBEDDING_LAYER_SIZE, NUM_LAYERS, NUM_HIDDEN_NEURONS, tp_dropout, str(layer_norm), key_covariables)

    results_filename = "results_"  + model_filename.replace("_parameters.pth", ".pkl")

    # ================== TREINO MODELO FIRST 46 MIN ==================
    # inicializa com pesos do modelo total
    token_model_46m.load_state_dict(token_model.state_dict())

    token_model_46m, trnloss, devloss = train_num_ngram_model(
        model = token_model,
        train_token_data = trn_tkn[:, -CONTEXT_SIZE:],
        train_numerical_data = trn_num[:, index_covariables],
        train_targets = trn_y,
        train_mask_data = trn_mask,
        batch_size = BATCH_SIZE,
        n_epochs = EPOCHS,
        lr = 0.002,
        val_token_data = dev_tkn[:, -CONTEXT_SIZE:],
        val_numerical_data = dev_num[:, index_covariables],
        val_targets = dev_y,
        val_mask_data = dev_mask,
        params_optimizer = {'factor': 0.5, 'threshold': 0, 'patience': 1},
        device = device)

    df_importance = permutation_importance_ngram(
        model = token_model_46m,
        target = dev_y,
        tokens = dev_tkn[:, -CONTEXT_SIZE:],
        numerical_features = dev_num[:, index_covariables],
        names_numerical_features = covariables,
        num_repeats = 10,
        mask_data = dev_mask)

    print(df_importance)

    output = token_model_46m(trn_tkn[:, -CONTEXT_SIZE:], trn_num[:, index_covariables])
    output = output.masked_fill(trn_mask, -1e9)
    baseline_loss = F.cross_entropy(output, trn_y).item()

    print(f"train data loss: {baseline_loss}")

    torch.save(token_model_46m.state_dict(), models_folder + model_filename)
    dict_results = {
        "class_model": token_model_46m.__class__.__name__,
        "model_filename": model_filename,
        "token_context_size": CONTEXT_SIZE,
        "token_embedding_dim": EMBEDDING_LAYER_SIZE,
        "training_epochs": EPOCHS,
        "numerical_covariables": covariables,
        "training_batch_loss": np.asarray(trnloss),
        "train_loss": baseline_loss,
        "dev_loss": np.asarray(devloss),
        "num_parameters": get_num_parameters(token_model_46m),
        "variable_importance": df_importance}

    with open(models_folder + results_filename, 'wb') as f:
        pickle.dump(dict_results, f)

    #######################################################################
    # ================== MODELO  2 MINUTOS (FINE-TUNING) ==================
    df_ = df_token_features.query("season == @season & season_split == 'train' & ind_last_2_minutes_game == 1")

    trn_tkn = torch.tensor(np.asarray(df_["token_features"].to_list()), dtype = torch.int64, device = device)
    trn_num = torch.tensor(np.asarray(df_["numerical_features"].to_list()), dtype = torch.float32, device = device)
    trn_y   = torch.tensor(np.asarray(df_["target_tokens"].to_list()), dtype = torch.int64, device = device)
    trn_mask = torch.tensor(build_logit_mask_ngram_models(df_), dtype = torch.bool, device = device)

    df_ = df_token_features.query("season == @season & season_split == 'dev' & ind_last_2_minutes_game == 1")

    dev_tkn = torch.tensor(np.asarray(df_["token_features"].to_list()), dtype = torch.int64, device = device)
    dev_num = torch.tensor(np.asarray(df_["numerical_features"].to_list()), dtype = torch.float32, device = device)
    dev_y   = torch.tensor(np.asarray(df_["target_tokens"].to_list()), dtype = torch.int64, device = device)
    dev_mask = torch.tensor(build_logit_mask_ngram_models(df_), dtype = torch.bool, device = device)

    del df_

    token_model_2m = NGramModel_ExtraInfo(
        context_size = CONTEXT_SIZE,
        vocab_size = VOCAB_SIZE,
        embedding_dim = EMBEDDING_LAYER_SIZE,
        numerical_input_dim = len(index_covariables),
        num_hidden_neurons = NUM_HIDDEN_NEURONS,
        num_layers = NUM_LAYERS,
        activation = nn.GELU,
        dropout_rate = dropout_rate,
        use_layer_norm = layer_norm)

    str_season = season.replace("-", "")

    model_filename = model_mask_name_2min\
    .format(token_model_2m.__class__.__name__, str_season, CONTEXT_SIZE, EMBEDDING_LAYER_SIZE, NUM_LAYERS, NUM_HIDDEN_NEURONS, tp_dropout, str(layer_norm), key_covariables)

    results_filename = "results_"  + model_filename.replace("_parameters.pth", ".pkl")

    # ================== TREINO MODELO LAST 2 MIN ==================
    # inicializa com pesos do modelo total
    token_model_2m.load_state_dict(token_model.state_dict())

    token_model_2m, trnloss, devloss = train_num_ngram_model(
        model = token_model_2m,
        train_token_data = trn_tkn[:, -CONTEXT_SIZE:],
        train_numerical_data = trn_num[:, index_covariables],
        train_targets = trn_y,
        train_mask_data = trn_mask,
        batch_size = BATCH_SIZE,
        n_epochs = EPOCHS,
        lr = 0.001,
        val_token_data = dev_tkn[:, -CONTEXT_SIZE:],
        val_numerical_data = dev_num[:, index_covariables],
        val_targets = dev_y,
        val_mask_data = dev_mask,
        params_optimizer = {'factor': 0.5, 'threshold': 0, 'patience': 1},
        device = device)

    df_importance = permutation_importance_ngram(
        model = token_model_2m,
        target = dev_y,
        tokens = dev_tkn[:, -CONTEXT_SIZE:],
        numerical_features = dev_num[:, index_covariables],
        names_numerical_features = covariables,
        num_repeats = 10,
        mask_data = dev_mask)


    print(df_importance)

    output = token_model_2m(trn_tkn[:, -CONTEXT_SIZE:], trn_num[:, index_covariables])
    output = output.masked_fill(trn_mask, -1e9)
    baseline_loss = F.cross_entropy(output, trn_y).item()

    print(f"train data loss: {baseline_loss}")

    torch.save(token_model_2m.state_dict(), models_folder + model_filename)
    dict_results = {
        "class_model": token_model_2m.__class__.__name__,
        "model_filename": model_filename,
        "token_context_size": CONTEXT_SIZE,
        "token_embedding_dim": EMBEDDING_LAYER_SIZE,
        "training_epochs": EPOCHS,
        "numerical_covariables": covariables,
        "training_batch_loss": np.asarray(trnloss),
        "train_loss": baseline_loss,
        "dev_loss": np.asarray(devloss),
        "num_parameters": get_num_parameters(token_model_2m),
        "variable_importance": df_importance}

    with open(models_folder + results_filename, 'wb') as f:
        pickle.dump(dict_results, f)

    clear_output(wait = True)



160
('2022-23', 5, 64, 50, 5, 0.3, True, 2)
Baseline model loss:  2.8482571 | Val Loss: 2.84819 | LR: 2.50e-04
                            feature  count    mean     std      min      25%      50%      75%      max  baseline     perc   percacm
4                           token_1   10.0  9.0223  0.0301  11.8111  11.8517  11.8798  11.8958  11.8999    2.8482  92.4690   92.4690
3                           token_2   10.0  0.4182  0.0064   3.2567   3.2614   3.2681   3.2720   3.2724    2.8482   4.2861   96.7552
23                  ind_last_second   10.0  0.0792  0.0013   2.9253   2.9267   2.9276   2.9278   2.9301    2.8482   0.8117   97.5669
2                           token_3   10.0  0.0688  0.0020   2.9144   2.9153   2.9173   2.9181   2.9198    2.8482   0.7053   98.2721
44                  ball_possession   10.0  0.0451  0.0025   2.8892   2.8921   2.8932   2.8955   2.8966    2.8482   0.4624   98.7345
22               ind_last_5_seconds   10.0  0.0352  0.0004   2.8824   2.8832   2.8835   2.8